# 32. Metaclasses and class creation

**Level:** Advanced

**Learning outcome:** Distinguish instances, classes, and metaclasses and decide when an ordinary factory is sufficient.

**Prerequisites:** 06, 21, and 23: MRO, descriptors, __new__, and subclass hooks.

[Learning-path index](00_START_HERE_OOP_Learning_Path.ipynb) | [Previous](31_Attribute_Lookup_Introspection_and_Dynamic_Access.ipynb) | [Next notebook](33_Async_Context_Managers_Iteration_and_Cancellation.ipynb)

## How to study this notebook
1. Read the mental model and explanation before executing a code cell.
2. Predict the output, then run the cell and compare the stored result.
3. Trace constructor calls, attribute reads/writes, returned values, and exception paths.
4. Change one input, predict the difference, and keep meaningful assertions.
5. Complete the practice prompts and answer the final checkpoint aloud.

**Run independently:** every worked code cell includes its own imports, definitions, and input data. You do not need to run another notebook first. Use Python 3.11+ and a Python Jupyter kernel. Only the standard library is used. Async cells use top-level await, not asyncio.run inside Jupyter.

**Offline safety:** all network states, commands, approvals, and failures are simulations. The code does not connect to devices, use credentials, write device configuration, or need external input files. Teaching limits and command strings are not vendor specifications. Assertions are learning checks, not production input validation.

## Mental model
A class is itself an object. A normal Device instance has Device as its type; Device usually has
type as its type. A metaclass customizes class creation. Class bodies execute to prepare a
namespace, and class-creation machinery turns that namespace into the class object.

`type(name, bases, namespace)` can construct an ordinary class dynamically. It is useful to
understand the mechanism, not a reason to generate a class for every inventory row. Instances
represent varying data; classes represent meaningful behavior and contracts.

A metaclass's __new__ can inspect the class namespace before delegating to type.__new__.
An instance __new__ instead controls instance creation. Keep those two levels distinct.
ABCMeta already supplies abstract-class behavior; combining unrelated metaclasses can cause
conflicts. Do not solve a speculative plugin problem by stacking metaclasses.

Prefer an explicit register call, a class decorator, or __init_subclass__ when it expresses the
requirement. Use a custom metaclass only when the application has a demonstrated class-level
invariant that the simpler mechanism cannot express adequately.

## Worked example 1: Instances, classes, and metaclasses are different levels

**Problem:** Inspect creation levels without relying on printed memory addresses.

### Step-by-step explanation
1. Construct an ordinary instance through its class.
2. Ask for the instance's type, then the class object's type.
3. Use identity checks for the actual type objects.
4. Print stable names rather than implementation-specific object addresses.

**Expected output:**
```text
Device type
```

**Common mistake / limitation:** isinstance follows subclass relationships; type(x) is T asks for an exact type.

**Verification:** the next code cell asserts results, state, or expected rejection. Compare those checks with the explanation, not just the printed text.

In [1]:
class Device:
    def __init__(self, hostname):
        self.hostname = hostname

device = Device("edge-01")
assert type(device) is Device
assert type(Device) is type
assert isinstance(Device, type)
assert isinstance(device, Device)
print(type(device).__name__, type(Device).__name__)

Device type


### Practice 1

Create a Router subclass and compare isinstance(router, Device) with type(router) is Device.

Predict the result before executing your extension. Keep one normal case and one edge/rejection case. State whether the extension changes identity, ownership, input validation, or error propagation.

## Worked example 2: Dynamic class creation with type

**Problem:** Understand the dynamic class constructor while keeping instance data separate.

### Step-by-step explanation
1. Supply a class name, a tuple of bases, and a namespace mapping.
2. The function in that namespace behaves as a normal bound method.
3. Create two instances for varying hostnames rather than two generated classes.
4. Verify both instances share the type while keeping separate data.

**Expected output:**
```text
lab:edge-01 lab:edge-02
```

**Common mistake / limitation:** Do not generate executable code with eval from untrusted inventory fields.

**Verification:** the next code cell asserts results, state, or expected rejection. Compare those checks with the explanation, not just the printed text.

In [2]:
def describe(self):
    return f"{self.role}:{self.hostname}"

OfflineDevice = type("OfflineDevice", (), {"role": "lab", "describe": describe})
first, second = OfflineDevice(), OfflineDevice()
first.hostname, second.hostname = "edge-01", "edge-02"
assert first.describe() == "lab:edge-01"
assert second.describe() == "lab:edge-02"
assert first.__class__ is second.__class__
print(first.describe(), second.describe())

lab:edge-01 lab:edge-02


### Practice 2

Rewrite this as an ordinary class statement and compare clarity.

Predict the result before executing your extension. Keep one normal case and one edge/rejection case. State whether the extension changes identity, ownership, input validation, or error propagation.

## Worked example 3: Metaclass enforces a narrow class-definition contract

**Problem:** Reject a plugin class with missing declared metadata before constructing any instance.

### Step-by-step explanation
1. The root Plugin has no explicit bases in its class definition, so this local policy exempts it.
2. A child class must declare a nonempty plugin_name in its own namespace.
3. Delegate valid construction to type.__new__ through super.
4. Catch the expected error around the invalid class statement itself, not instance creation.

**Expected output:**
```text
plugin_name must be declared
mtu
```

**Common mistake / limitation:** This is intentionally strict and does not inherit plugin_name; __init_subclass__ is often simpler for this exact rule.

**Verification:** the next code cell asserts results, state, or expected rejection. Compare those checks with the explanation, not just the printed text.

In [3]:
class NamedPluginMeta(type):
    def __new__(metaclass, name, bases, namespace):
        if bases:
            value = namespace.get("plugin_name")
            if not isinstance(value, str) or not value.strip():
                raise TypeError("plugin_name must be declared")
        return super().__new__(metaclass, name, bases, namespace)

class Plugin(metaclass=NamedPluginMeta):
    pass

class MtuPlugin(Plugin):
    plugin_name = "mtu"

assert isinstance(MtuPlugin, NamedPluginMeta)
try:
    class InvalidPlugin(Plugin):
        pass
except TypeError as error:
    assert str(error) == "plugin_name must be declared"
    print(str(error))
else:
    raise AssertionError("invalid plugin class created")
print(MtuPlugin.plugin_name)

plugin_name must be declared
mtu


### Practice 3

Implement the same rule with __init_subclass__ and explain which version you would maintain.

Predict the result before executing your extension. Keep one normal case and one edge/rejection case. State whether the extension changes identity, ownership, input validation, or error propagation.

## Check your understanding: Metaclasses and class creation

**You should now be able to:** Distinguish instances, classes, and metaclasses and decide when an ordinary factory is sufficient.

1. Explain which object owns each piece of state and which operations can change it.
2. Walk through a normal result and an invalid-input or failure path without running the code.
3. Name one invariant, one useful assertion, and one production assumption not proved by the fake.
4. Implement one extension prompt, including a test that would fail if your new behavior were wrong.
5. Explain when a function or a dataclass would be simpler than another subclass or framework.

**Do not confuse:** a successful local assertion with real device compatibility; missing data with a healthy result; underscore naming with access control; type annotations with runtime validation; or local rollback with a multi-device transaction.

## Official reference reading
- `https://docs.python.org/3.13/reference/datamodel.html`
- `https://docs.python.org/3.13/library/functions.html`
- `https://docs.python.org/3.13/library/abc.html`

References describe Python contracts. They do not certify any vendor command or network policy.